# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)

num_cols = [
    'HAARBRADES', 'HANOARBRAD', 'SuperficieTotalForestal',
    'HANOFOREST', 'SuperficieAgricola',
    'OtrasSuperficiesNoforestales', 'SuperficieTotalNoForestal'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['SuperficieTotal'] = (
    df['SuperficieTotalForestal'].fillna(0)
    + df['SuperficieTotalNoForestal'].fillna(0)
)


def any_des_de_data(serie):
    dates = pd.to_datetime(serie, dayfirst=True, errors='coerce', format='mixed')
    return dates.dt.year


any_1 = any_des_de_data(df['Data incendi'])
any_2 = any_des_de_data(df['Data incendi'])
any_campania = df['Campania'].astype(str).str.extract(r'(\d{4})\s*$')[0].astype(float)
df['Any'] = any_1.combine_first(any_2).combine_first(any_campania)

df = df.dropna(subset=['Nom comarca', 'Any']).copy()
df['Any'] = df['Any'].astype(int)
df = df[(df['Any'] >= 1998) & (df['Any'] <= 2024)]

# Unifiquem causes buides o "Desconeguda" amb la categoria "Desconocida" real del CSV
_causes_valides = df['Causa'].dropna().astype(str).str.strip()
_candidata_desconeguda = next(
    (c for c in _causes_valides.unique() if 'esconoc' in c.lower()),
    next((c for c in _causes_valides.unique() if 'esconeg' in c.lower()), 'Desconocida')
)
df['Causa'] = df['Causa'].fillna(_candidata_desconeguda).astype(str).str.strip()
df['Causa'] = df['Causa'].replace({
    'nan': _candidata_desconeguda,
    '': _candidata_desconeguda,
    'Desconeguda': _candidata_desconeguda,
    'desconeguda': _candidata_desconeguda
})

for c in ['Nom comarca', 'Nom vegueria']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

correccions_comarca = {
    'Aran': "Val d'Aran",
    'Montisà': 'Montsià',
    "Val d'Aran": "Val d'Aran"
}
df['Nom comarca'] = df['Nom comarca'].replace(correccions_comarca)

top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [1]:
# --- Classes de mida de l'incendi (Q4) ---
classes_mida = ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
df['ClasseMida'] = pd.cut(
    df['SuperficieTotal'],
    bins=[-0.01, 1, 10, 100, 500, 1e9],
    labels=classes_mida
)
mida_scale = alt.Scale(
    domain=classes_mida,
    scheme=alt.SchemeParams(name='inferno', extent=[0.12, 0.92]),
    reverse=True
)

NameError: name 'pd' is not defined

# %% ===================================================================
# Q4 FINAL (trienni) · Hectàrees cremades (barra mirall) + Superfície Forestal (donut)
#                       + Nombre d'incendis per magnitud (boles)
# =======================================================================
# SUPOSICIONS: aquestes variables ja existeixen de cel·les anteriors del teu
# notebook i NO es redefineixen aquí:
#   - df               -> DataFrame amb 'Any', 'SuperficieTotal', 'ClasseMida',
#                          'SuperficieTotalForestal', 'SuperficieTotalNoForestal',
#                          'Nom municipi', 'Nom comarca'
#   - classes_mida     -> llista ordenada ASCENDENT, p.ex.
#                          ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
#   - mida_scale       -> alt.Scale(domain=classes_mida, range=[...]) colorblind-safe
#   - titol_rt(t, sub) -> helper que retorna alt.TitleParams(t, subtitle=sub, ...)
#
# Si alguna d'aquestes no existeix, el script fa un "fallback" senzill.
# -----------------------------------------------------------------------
import pandas as pd
import altair as alt

if 'titol_rt' not in globals():
    def titol_rt(t, sub):
        return alt.TitleParams(t, subtitle=sub, fontSize=16, subtitleFontSize=12, anchor='start')

# ---------------------------------------------------------------------
# 0) REAGRUPACIÓ TEMPORAL: TRIENNIS (abans era cada 4 anys)
# ---------------------------------------------------------------------
ANY_MIN = int(df['Any'].min())
MIDA_PERIODE = 3  # trienni


def assigna_trienni(any_valor, any_min=ANY_MIN, mida=MIDA_PERIODE):
    idx = (any_valor - any_min) // mida
    inici = any_min + idx * mida
    fi = inici + mida - 1
    return f"{inici}-{fi}"


df['PeriodeQ4'] = df['Any'].apply(assigna_trienni)
labels_q4 = sorted(df['PeriodeQ4'].unique(), key=lambda p: int(p.split('-')[0]))

# ---------------------------------------------------------------------
# 1) DADES — Hectàrees cremades per classe de mida i trienni (valors ABSOLUTS)
# ---------------------------------------------------------------------
df_q4_mida_raw = (
    df.groupby(['PeriodeQ4', 'ClasseMida'], as_index=False, observed=True)
    .agg(Superficie=('SuperficieTotal', 'sum'), NumFocs=('SuperficieTotal', 'count'))
    .rename(columns={'PeriodeQ4': 'Periode'})
)

parts_abs = []
for p in labels_q4:
    sub = (
        df_q4_mida_raw[df_q4_mida_raw['Periode'] == p]
        .set_index('ClasseMida').reindex(classes_mida).reset_index()
    )
    sub['Superficie'] = sub['Superficie'].fillna(0)
    sub['NumFocs'] = sub['NumFocs'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['pct'] = sub['Superficie'] / tot if tot else 0
    # Ordre ascendent (classes_mida): les classes petites queden a prop del 0
    # (centre) i les grans arriben fins a la punta llunyana -> reforça la paradoxa
    sub['x1_abs'] = sub['Superficie'].cumsum()
    sub['x0_abs'] = sub['x1_abs'] - sub['Superficie']
    sub['x_mig_abs'] = (sub['x0_abs'] + sub['x1_abs']) / 2
    sub['Periode'] = p
    sub['Total_Periode'] = tot
    sub['label_pct'] = sub['pct'].apply(lambda v: f"{round(v * 100)}%" if v >= 0.05 else "")
    parts_abs.append(sub)
df_q4_mida_abs = pd.concat(parts_abs, ignore_index=True)

max_ha_periode = df_q4_mida_abs.groupby('Periode')['x1_abs'].max().max()

totals_periode = df_q4_mida_abs.groupby('Periode', as_index=False)['Total_Periode'].first()
totals_periode['label_total'] = totals_periode['Total_Periode'].apply(lambda v: f"{v:,.0f} ha")

# Escala X "mirall": el 0 queda a la dreta (costat del centre) i els valors
# s'allunyen cap a l'esquerra. Format sense notació científica (',.0f').
X_SCALE_ABS = alt.Scale(domain=[max_ha_periode * 1.12, 0], nice=False)

# ---------------------------------------------------------------------
# 2) DADES — Superfície Forestal vs No Forestal (donut)
# ---------------------------------------------------------------------
df_q4_tipus_raw = (
    df.groupby('PeriodeQ4', as_index=False, observed=True)
    .agg(Forestal=('SuperficieTotalForestal', 'sum'), NoForestal=('SuperficieTotalNoForestal', 'sum'))
    .rename(columns={'PeriodeQ4': 'Periode'})
    .melt(id_vars='Periode', value_vars=['Forestal', 'NoForestal'], var_name='Tipus', value_name='Superficie')
)
tipus_domain = ['Forestal', 'NoForestal']
tipus_scale = alt.Scale(domain=tipus_domain, range=['#1b7837', '#b35806'])

parts_tipus = []
for p in labels_q4:
    sub = df_q4_tipus_raw[df_q4_tipus_raw['Periode'] == p].set_index('Tipus').reindex(tipus_domain).reset_index()
    sub['Superficie'] = sub['Superficie'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['pct'] = sub['Superficie'] / tot if tot else 0
    sub['Periode'] = p
    # Com a mínim la verda (Forestal) sempre porta el percentatge
    sub['label_pct'] = sub.apply(
        lambda r: f"{round(r['pct'] * 100)}%" if (r['Tipus'] == 'Forestal' or r['pct'] >= 0.12) else "", axis=1
    )
    parts_tipus.append(sub)
df_q4_tipus = pd.concat(parts_tipus, ignore_index=True)

# ---------------------------------------------------------------------
# 3) DADES — Boles "resta" (<500 ha, agrupades: 1 bola = 20 focs)
# ---------------------------------------------------------------------
GROUP_SIZE_Q4 = 20
COLS_RESTA_Q4 = 12  # compacte, perquè càpiga en una sola pantalla
classes_resta = [c for c in classes_mida if c != '>500 ha']


def boles_resta_periode(subdf, periode):
    files = []
    idx_global = 0
    for classe in reversed(classes_resta):
        n = int((subdf['ClasseMida'] == classe).sum())
        if n > 0 and classe == '100-500 ha':
            n_boles = max(1, int(round(n / GROUP_SIZE_Q4)))
        else:
            n_boles = int(n // GROUP_SIZE_Q4)
        for _ in range(n_boles):
            files.append({
                'Periode': periode, 'ClasseMida': classe,
                'col': idx_global % COLS_RESTA_Q4, 'fila': idx_global // COLS_RESTA_Q4,
                'Incendis_Reals': n,
                'Equivalencia': f"1 bola = {GROUP_SIZE_Q4} incendis ({classe})"
            })
            idx_global += 1
    return pd.DataFrame(files)


df_boles_resta = pd.concat(
    [boles_resta_periode(df[df['PeriodeQ4'] == p], p) for p in labels_q4], ignore_index=True
)
max_fila_resta = max(int(df_boles_resta['fila'].max()), 2)

# ---------------------------------------------------------------------
# 4) DADES — Boles GIF individuals (>500 ha, 1 bola = 1 incendi), màx 4x3
# ---------------------------------------------------------------------
COLS_GIF_Q4 = 4


def boles_gif_individuals(subdf, periode):
    files = []
    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for i, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        files.append({
            'Periode': periode, 'ClasseMida': '>500 ha',
            'col': i % COLS_GIF_Q4, 'fila': i // COLS_GIF_Q4,
            'Nom_Comarca': r['Nom comarca'], 'Ubicacio': f"{mun} ({int(r['Any'])})",
            'Any': int(r['Any']), 'ha_real': float(r['SuperficieTotal'])
        })
    return pd.DataFrame(files)


df_boles_gif = pd.concat(
    [boles_gif_individuals(df[df['PeriodeQ4'] == p], p) for p in labels_q4], ignore_index=True
)
max_fila_gif = max(int(df_boles_gif['fila'].max()) if len(df_boles_gif) else 2, 2)  # fins a 3 files (0,1,2)

# ---------------------------------------------------------------------
# 5) AMPLADES (barra x1.4 respecte l'original) i alçades
# ---------------------------------------------------------------------
ALCADA_BARRA = 46
W_BAR_ABS = 430          # <-- si la teva W_BAR_ABS original era, p.ex., 300, posa 300*1.4
W_ANY_Q4 = 78
W_DONUT = 150
W_GIF = 130
W_RESTA = 210
ESPAI = 8

# ---------------------------------------------------------------------
# 6) CAPÇALERES (títols de secció + llegendes)
# ---------------------------------------------------------------------
tit_bar = alt.Chart(pd.DataFrame({'t': ["Hectàrees cremades segons magnitud de l'incendi"]})).mark_text(
    fontSize=13, fontWeight='bold', align='left', color='#111111'
).encode(text='t:N').properties(width=W_BAR_ABS, height=1)

tit_any = alt.Chart(pd.DataFrame({'t': ['']})).mark_text().encode(text='t:N').properties(width=W_ANY_Q4, height=1)

tit_donut = alt.Chart(pd.DataFrame({'t': ['Superfície Forestal cremada']})).mark_text(
    fontSize=13, fontWeight='bold', align='left', color='#111111'
).encode(text='t:N').properties(width=W_DONUT, height=1)

tit_boles = alt.Chart(pd.DataFrame({'t': ["Nombre d'incendis per magnitud"]})).mark_text(
    fontSize=13, fontWeight='bold', align='left', color='#111111'
).encode(text='t:N').properties(width=W_GIF + W_RESTA + ESPAI, height=1)

fila_titols = alt.hconcat(tit_bar, tit_any, tit_donut, tit_boles, spacing=ESPAI)

llegenda_mida = alt.Chart(pd.DataFrame({'ClasseMida': classes_mida})).mark_circle(size=150, opacity=0).encode(
    color=alt.Color(
        'ClasseMida:N', scale=mida_scale, title="Magnitud:",
        legend=alt.Legend(orient='top', direction='horizontal', columns=3,
                           titleOrient='left', titleFontSize=11, labelFontSize=10,
                           symbolType='circle', symbolSize=150, symbolOpacity=1)
    )
).properties(width=W_BAR_ABS, height=1)

cap_trienni = alt.Chart(pd.DataFrame({'t': ['TRIENNI']})).mark_text(
    fontSize=12, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=W_ANY_Q4, height=1)

llegenda_tipus = alt.Chart(pd.DataFrame({'Tipus': tipus_domain})).mark_circle(size=150, opacity=0).encode(
    color=alt.Color('Tipus:N', scale=tipus_scale, title="Cobertura:",
                     legend=alt.Legend(orient='top', direction='horizontal', titleOrient='left',
                                        titleFontSize=11, labelFontSize=10, symbolType='circle', symbolSize=150))
).properties(width=W_DONUT, height=1)

llegenda_boles = alt.Chart(pd.DataFrame({
    't': ["① 1 bola = 20 incendis agrupats   ② ⬤ fosca = Gran Incendi Forestal individual (>500 ha)"]
})).mark_text(fontSize=10.5, align='left', color='#444444').encode(text='t:N').properties(
    width=W_GIF + W_RESTA + ESPAI, height=1
)

fila_llegendes = alt.hconcat(llegenda_mida, cap_trienni, llegenda_tipus, llegenda_boles, spacing=ESPAI)

# ---------------------------------------------------------------------
# 7) UNA FILA PER TRIENNI
# ---------------------------------------------------------------------
files_final = [fila_titols, fila_llegendes]

for idx_p, p in enumerate(labels_q4):
    is_last = idx_p == len(labels_q4) - 1

    # --- 7.1 Barra mirall (hectàrees absolutes, 0 a la dreta) ---
    sub_bar = df_q4_mida_abs[df_q4_mida_abs['Periode'] == p]
    total_p_df = totals_periode[totals_periode['Periode'] == p]

    c_bar = (
        alt.Chart(sub_bar).mark_bar(height=ALCADA_BARRA).encode(
            x=alt.X(
                'x0_abs:Q', scale=X_SCALE_ABS,
                axis=alt.Axis(title='Superfície total cremada (ha)', format=',.0f') if is_last else None
            ),
            x2='x1_abs:Q',
            color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
            tooltip=[
                alt.Tooltip('Periode:N', title='Trienni'),
                alt.Tooltip('ClasseMida:N', title='Magnitud'),
                alt.Tooltip('Superficie:Q', title='Hectàrees cremades (ha)', format=',.1f'),
                alt.Tooltip('NumFocs:Q', title="Nombre d'incendis", format=',')
            ]
        )
        + alt.Chart(sub_bar[sub_bar['pct'] >= 0.05]).mark_text(
            color='white', fontSize=10.5, fontWeight='bold', baseline='middle'
        ).encode(x=alt.X('x_mig_abs:Q', scale=X_SCALE_ABS), text='label_pct:N')
        + alt.Chart(total_p_df).mark_text(
            align='right', dx=-6, fontSize=10.5, fontWeight='bold', color='#111111', baseline='middle'
        ).encode(x=alt.X('Total_Periode:Q', scale=X_SCALE_ABS), text='label_total:N')
    ).properties(width=W_BAR_ABS, height=ALCADA_BARRA + 8)

    # --- 7.2 Trienni (centre) ---
    c_any = alt.Chart(pd.DataFrame({'t': [p]})).mark_text(
        fontSize=16, fontWeight='bold', align='center', color='#111111'
    ).encode(text='t:N').properties(width=W_ANY_Q4, height=ALCADA_BARRA + 8)

    # --- 7.3 Donut Forestal vs No Forestal ---
    sub_t = df_q4_tipus[df_q4_tipus['Periode'] == p]
    arc = alt.Chart(sub_t).mark_arc(innerRadius=16, outerRadius=32, stroke='white', strokeWidth=1).encode(
        theta=alt.Theta('Superficie:Q', stack=True),
        color=alt.Color('Tipus:N', scale=tipus_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('Tipus:N', title='Cobertura'),
            alt.Tooltip('Superficie:Q', title='Hectàrees (ha)', format=',.1f'),
            alt.Tooltip('pct:Q', title='Percentatge', format='.1%')
        ]
    )
    text_arc = alt.Chart(sub_t).mark_text(radius=24, fontSize=9.5, fontWeight='bold', color='white').encode(
        theta=alt.Theta('Superficie:Q', stack=True), text='label_pct:N'
    )
    c_donut = (arc + text_arc).properties(width=W_DONUT, height=ALCADA_BARRA + 8)

    # --- 7.4 Boles GIF individuals (>500 ha), sense requadre, 4 columnes ---
    sub_g = df_boles_gif[df_boles_gif['Periode'] == p]
    c_boles_gif = alt.Chart(sub_g).mark_circle(
        size=130, color='#1a1a1a', opacity=0.95, stroke='#d73027', strokeWidth=1.1
    ).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.5, COLS_GIF_Q4 - 0.5], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.5, max_fila_gif + 0.5], reverse=True, nice=False), axis=None),
        tooltip=[
            alt.Tooltip('Ubicacio:N', title='Gran Incendi Forestal (GIF)'),
            alt.Tooltip('Nom_Comarca:N', title='Comarca'),
            alt.Tooltip('Any:Q', title='Any'),
            alt.Tooltip('ha_real:Q', title='Hectàrees cremades (ha)', format=',.1f')
        ]
    ).properties(width=W_GIF, height=ALCADA_BARRA + 8)

    # --- 7.5 Boles "resta" agrupades (1 bola = 20 focs) ---
    sub_r = df_boles_resta[df_boles_resta['Periode'] == p]
    c_boles_resta = alt.Chart(sub_r).mark_circle(
        size=48, opacity=0.95, stroke='white', strokeWidth=0.4
    ).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.5, COLS_RESTA_Q4 - 0.5], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.5, max_fila_resta + 0.5], reverse=True, nice=False), axis=None),
        color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('ClasseMida:N', title='Categoria'),
            alt.Tooltip('Incendis_Reals:Q', title="Total incendis d'aquesta classe", format=','),
            alt.Tooltip('Equivalencia:N', title='Escala')
        ]
    ).properties(width=W_RESTA, height=ALCADA_BARRA + 8)

    files_final.append(
        alt.hconcat(c_bar, c_any, c_donut, c_boles_gif, c_boles_resta, spacing=ESPAI)
        .resolve_scale(color='independent')
    )

# ---------------------------------------------------------------------
# 8) COMPOSICIÓ FINAL
# ---------------------------------------------------------------------
q4_final_trienni = alt.vconcat(*files_final, spacing=3).properties(
    title=titol_rt(
        "Q4 · Evolució de la distribució dels incendis forestals a Catalunya per trienni (1998–2024)",
        "Hectàrees cremades per magnitud (barra mirall), % superfície forestal vs. no forestal i nombre d'incendis per magnitud"
    )
).configure_view(strokeWidth=0).configure_axis(grid=False)

q4_final_trienni.save('Q4_final_trienni.html')
